In [ ]:
 
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical

In [ ]:
# ===============================
# 1. Cargar datos procesados
# ===============================

PROJECT_ROOT = Path().resolve().parent
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# Cargar split de entrenamiento
X_train = np.load(PROCESSED_PATH / "train" / "X.npy")
y_train = np.load(PROCESSED_PATH / "train" / "y.npy")

# Cargar validación
X_val = np.load(PROCESSED_PATH / "val" / "X.npy")
y_val = np.load(PROCESSED_PATH / "val" / "y.npy")

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Pixel range:", X_train.min(), X_train.max())

Shape: (9459, 128, 128, 3)
Pixel range: 0.0 1.0


In [ ]:
# ===============================
# 2. Parámetros principales
# ===============================

input_shape = (128, 128, 3)
latent_dim = 16   # pequeño para empezar simple

### Encoder

In [ ]:
# ===============================
# 3. Encoder
# ===============================

encoder_inputs = layers.Input(shape=input_shape)

x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(encoder_inputs)  # 64x64
x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)              # 32x32
x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)             # 16x16

x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)

# Media y log varianza
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

In [ ]:
# ===============================
# 4. Sampling
# ===============================

def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])

In [ ]:
# ===============================
# 5. Decoder
# ===============================

decoder_inputs = layers.Input(shape=(latent_dim,))

x = layers.Dense(16*16*128, activation="relu")(decoder_inputs)
x = layers.Reshape((16,16,128))(x)

x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)  # 32x32
x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)   # 64x64
x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)   # 128x128

decoder_outputs = layers.Conv2D(3, 3, padding="same", activation="sigmoid")(x)

decoder = Model(decoder_inputs, decoder_outputs, name="decoder")

In [ ]:
# ===============================
# 6. Modelo VAE completo
# ===============================

encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

z_mean, z_log_var, z = encoder(encoder_inputs)
reconstructed = decoder(z)

vae = Model(encoder_inputs, reconstructed, name="VAE")

In [ ]:
# ===============================
# 7. Función de pérdida
# ===============================

reconstruction_loss = tf.reduce_mean(
    tf.keras.losses.binary_crossentropy(encoder_inputs, reconstructed)
) * 128 * 128

kl_loss = -0.5 * tf.reduce_mean(
    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
)

vae.add_loss(reconstruction_loss + kl_loss)
vae.compile(optimizer=tf.keras.optimizers.Adam())

Epoch 1/20


In [ ]:
# ===============================
# 8. Entrenamiento
# ===============================

history = vae.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, X_val)
)

Training Beta = 0.5


ValueError: Received an invalid value for `units`, expected a positive integer. Received: units=16384

In [ ]:
# ===============================
# 9. Visualizar reconstrucciones
# ===============================

import matplotlib.pyplot as plt

recon = vae.predict(X_val[:5])

plt.figure(figsize=(10,4))

for i in range(5):
    # Original
    plt.subplot(2,5,i+1)
    plt.imshow(X_val[i])
    plt.axis("off")
    
    # Reconstruida
    plt.subplot(2,5,i+6)
    plt.imshow(recon[i])
    plt.axis("off")

plt.show()

In [ ]:
# ===============================
# 10. Generar caras nuevas
# ===============================

random_latent = np.random.normal(size=(5, latent_dim))
generated_images = decoder.predict(random_latent)

plt.figure(figsize=(10,2))

for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(generated_images[i])
    plt.axis("off")

plt.show()